# 02 Run Single-Subject Gammatone-8 TRF

This notebook runs the first formal Eelbrain-main TRF model for one subject. 


In [ ]:
from pathlib import Path
import sys

def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / 'analysis' / 'trf_pipeline']
    for path in candidates:
        if (path / 'alice_eelbrain_main_experiment.py').exists():
            return path
    raise FileNotFoundError(f'Could not find alice_eelbrain_main_experiment.py from {start}')


PIPELINE_DIR = find_pipeline_dir()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from alice_eelbrain_main_experiment import TRF_OPTIONS, alice

SUBJECT = '01'
MODEL = 'gammatone-8'
RUN_TRF = True
STATE = {'subject': SUBJECT, 'raw': '0.5-20', 'epoch': 'story-segments', 'inv': ''}

print(f'Subject: {SUBJECT}')
print(f'Model: {MODEL}')
print(TRF_OPTIONS)


## Confirm Cache Path

This shows where the TRF result will be cached. It does not run the model.

In [ ]:
# path_only=True returns the Eelbrain-main cache path without fitting the TRF.
trf_path = alice.load_trf(MODEL, path_only=True, **STATE, **TRF_OPTIONS)
print(trf_path)


## Run TRF for One Subject

Set `RUN_TRF = True` above to execute this cell. This can take time because it filters EEG, loads predictors, fits the TRF, and writes a cached pipeline result.

In [ ]:
if RUN_TRF:
    result = alice.load_trf(MODEL, **STATE, **TRF_OPTIONS)
    display(result)
else:
    print('RUN_TRF is False; not estimating TRF yet.')


## Inspect Pipeline Output

After running the previous cell, this cell displays the result attributes that are most relevant for acoustic tracking.

In [ ]:
if 'result' in globals():
    print(type(result))
    for attr in ['r', 'residual', 'proportion_explained', 'h', 'h_scaled', 'partitions']:
        if hasattr(result, attr):
            value = getattr(result, attr)
            print(f'\n{attr}:')
            print(value)
else:
    print('No result object yet. Run the TRF cell first.')

## Load Subject-Level Dataset Output

`load_trfs()` converts the cached TRF result into an Eelbrain Dataset containing fit metrics like `r`, `z`, `residual`, and `det`.


In [ ]:
if RUN_TRF:
    ds = alice.load_trfs(SUBJECT, MODEL, raw='0.5-20', epoch='story-segments', inv='', **TRF_OPTIONS)
    display(ds)
    print(ds.keys())
else:
    print('Set RUN_TRF=True and run the TRF first before loading dataset output.')
